<a href="https://colab.research.google.com/github/arunpradeep-g/L4-Assignments/blob/main/02-telemetry_agent_llama_index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U \
    llama-index-core \
    llama-index-llms-google-genai \
    llama-index-embeddings-huggingface \
    llama-index-vector-stores-chroma \
    llama-index-utils-workflow \
    chromadb sentence-transformers

import llama_index.core
print("llama-index-core:", llama_index.core.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.0/165.0 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.

In [ ]:
import os
import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass
    os.environ["GOOGLE_API_KEY"] = key or getpass.getpass("Enter Gemini API key: ")

if not os.environ.get("GOOGLE_API_KEY"):
    raise ValueError("Please set GOOGLE_API_KEY first.")

print("GOOGLE_API_KEY loaded")

GOOGLE_API_KEY loaded


In [ ]:
from google import genai

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
available = [m.name.replace("models/", "") for m in client.models.list()]
flash = [m for m in available if "flash" in m and "embedding" not in m]

print("Flash-class models visible to this key:")
for name in flash:
    print(" -", name)

PREFERRED = ["gemini-2.5-flash", "gemini-2.0-flash", "gemini-flash-latest"]
MODEL = next((m for m in PREFERRED if m in available), flash[0] if flash else available[0])
print("\nUsing model:", MODEL)

Flash-class models visible to this key:
 - gemini-2.5-flash
 - gemini-2.5-flash-preview-tts
 - gemini-flash-latest
 - gemini-flash-lite-latest
 - gemini-2.5-flash-lite
 - gemini-2.5-flash-image
 - gemini-3-flash-preview
 - gemini-3.1-flash-lite-preview
 - gemini-3.1-flash-lite
 - gemini-3.1-flash-image-preview
 - gemini-3.1-flash-image
 - gemini-3.1-flash-lite-image
 - gemini-3.5-flash
 - gemini-3.5-flash-lite
 - gemini-omni-flash-preview
 - gemini-3.6-flash
 - gemini-3.7-flash
 - gemini-3.1-flash-tts-preview
 - gemini-2.5-flash-native-audio-latest
 - gemini-2.5-flash-native-audio-preview-09-2025
 - gemini-2.5-flash-native-audio-preview-12-2025
 - gemini-3.1-flash-live-preview

Using model: gemini-2.5-flash


In [ ]:
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = GoogleGenAI(
    model=MODEL,
    api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0.2,
)

# Local MiniLM embeddings - no embedding quota consumed, same model as the original.
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("LLM       :", Settings.llm.metadata.model_name)
print("Embeddings:", Settings.embed_model.model_name)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

LLM       : gemini-2.5-flash
Embeddings: sentence-transformers/all-MiniLM-L6-v2


In [ ]:
import chromadb
from llama_index.core import Document, StorageContext, VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore

docs = [
    Document(
        text="""
        Engine overheating diagnosis:
        If coolant temperature is above 105 C, check coolant level, radiator fan,
        thermostat, coolant pump, radiator blockage, and coolant leakage.
        If temperature is above 115 C, vehicle must be stopped safely.
        """,
        metadata={"topic": "engine_cooling"},
    ),
    Document(
        text="""
        Brake pressure safety:
        Normal brake hydraulic pressure should generally remain above safe threshold.
        If brake pressure falls below 55 bar, braking performance may be compromised.
        If brake pressure is critically low, escalate immediately and advise driver to stop.
        """,
        metadata={"topic": "brake_safety"},
    ),
    Document(
        text="""
        Immobilization policy:
        Immobilize only when the condition is safety critical and human approval is given,
        or when autonomous emergency policy permits immobilization after the vehicle reaches
        a safe stop condition.
        """,
        metadata={"topic": "immobilization"},
    ),
    Document(
        text="""
        Low diagnostic confidence:
        If confidence is below 0.70, retrieve additional maintenance documents and repeat diagnosis.
        Confidence improves when multiple symptoms match the same fault family.
        """,
        metadata={"topic": "confidence_policy"},
    ),
    Document(
        text="""
        Battery and electrical health:
        Battery voltage below 11.8 V while the engine is running indicates a charging system fault.
        Inspect alternator output, belt tension, and battery terminals.
        """,
        metadata={"topic": "electrical"},
    ),
]

COLLECTION = "automotive_telemetry_kb"
db = chromadb.PersistentClient(path="./chroma_auto_telemetry_li")

# Rebuild the collection so re-running this cell does not duplicate documents.
try:
    db.delete_collection(COLLECTION)
except Exception:
    pass

chroma_collection = db.get_or_create_collection(COLLECTION)
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(docs, storage_context=storage_context)
retriever = index.as_retriever(similarity_top_k=2)

hits = retriever.retrieve("brake pressure too low escalation")
print(f"Indexed {chroma_collection.count()} nodes. Smoke-test retrieval:")
for node in hits:
    print(f"  score={node.score:.3f} topic={node.metadata.get('topic')}")

Indexed 5 nodes. Smoke-test retrieval:
  score=0.416 topic=brake_safety
  score=0.189 topic=confidence_policy


In [ ]:
from llama_index.core.workflow import (
    Context,
    Event,
    StartEvent,
    StopEvent,
    Workflow,
    step,
)


class AnalyzeEvent(Event):
    """Initial state stored; ready for threshold analysis."""


class RetrieveCoolingEvent(Event):
    pass


class RetrieveBrakeEvent(Event):
    pass


class RetrieveGeneralEvent(Event):
    pass


class DiagnoseEvent(Event):
    """Context is loaded; ask the LLM. Re-emitted to retry."""


class RetrieveMoreEvent(Event):
    """Confidence too low; widen retrieval and loop back."""


class HumanApprovalEvent(Event):
    pass


class ImmobilizeEvent(Event):
    pass


class ReportEvent(Event):
    pass


print("events defined")

events defined


In [ ]:
import json


def _store(ctx: Context):
    return getattr(ctx, "store", ctx)


async def sget(ctx: Context, key: str, default=None):
    try:
        return await _store(ctx).get(key, default=default)
    except TypeError:
        return await _store(ctx).get(key)


async def sset(ctx: Context, key: str, value) -> None:
    await _store(ctx).set(key, value)


async def slog(ctx: Context, message: str) -> None:
    entries = await sget(ctx, "state_log", [])
    entries = list(entries) + [message]
    await sset(ctx, "state_log", entries)
    print("  [log]", message)


print("state helpers ready")

state helpers ready


In [ ]:
import asyncio
from typing import Any, Dict, List

HUMAN_APPROVAL_MODE = "prompt"  # "prompt" | "auto_yes" | "auto_no"


class TelemetryDiagnosticWorkflow(Workflow):
    # ---------------------------------------
    # NODE: STORE INITIAL STATE
    # ---------------------------------------
    @step
    async def store_vehicle_state(self, ctx: Context, ev: StartEvent) -> AnalyzeEvent:
        await sset(ctx, "vehicle_id", ev.vehicle_id)
        await sset(ctx, "telemetry", ev.telemetry)
        await sset(ctx, "state_log", [])
        await sset(ctx, "retrieved_docs", [])
        await sset(ctx, "risk_flags", [])
        await sset(ctx, "route", "")
        await sset(ctx, "diagnosis", "")
        await sset(ctx, "confidence", 0.0)
        await sset(ctx, "retry_count", 0)
        await sset(ctx, "safety_critical", False)
        await sset(ctx, "human_approval_required", False)
        await sset(ctx, "human_approved", False)
        await sset(ctx, "action", "")

        snapshot = {"vehicle_id": ev.vehicle_id, "telemetry": ev.telemetry}
        await slog(ctx, f"Stored initial vehicle state: {json.dumps(snapshot)}")
        return AnalyzeEvent()

    # ---------------------------------------
    # NODE + ROUTER: ANALYZE TELEMETRY, CHOOSE FAULT PATH
    # ---------------------------------------
    @step
    async def analyze_telemetry(
        self, ctx: Context, ev: AnalyzeEvent
    ) -> RetrieveCoolingEvent | RetrieveBrakeEvent | RetrieveGeneralEvent:
        t: Dict[str, Any] = await sget(ctx, "telemetry", {})
        flags: List[str] = []

        if t.get("engine_temp_c", 0) >= 105:
            flags.append("ENGINE_TEMP_HIGH")

        if t.get("brake_pressure_bar", 999) < 55:
            flags.append("BRAKE_PRESSURE_LOW")

        if t.get("vehicle_speed_kmph", 0) > 0 and t.get("brake_pressure_bar", 999) < 45:
            flags.append("DANGEROUS_BRAKE_WHILE_MOVING")

        safety_critical = (
            "DANGEROUS_BRAKE_WHILE_MOVING" in flags
            or t.get("engine_temp_c", 0) >= 115
            or t.get("brake_pressure_bar", 999) < 40
        )

        await sset(ctx, "risk_flags", flags)
        await sset(ctx, "safety_critical", safety_critical)
        await slog(
            ctx, f"Telemetry analyzed. Flags={flags}, safety_critical={safety_critical}"
        )

        if "BRAKE_PRESSURE_LOW" in flags or "DANGEROUS_BRAKE_WHILE_MOVING" in flags:
            return RetrieveBrakeEvent()
        if "ENGINE_TEMP_HIGH" in flags:
            return RetrieveCoolingEvent()
        return RetrieveGeneralEvent()

    # ---------------------------------------
    # RETRIEVAL NODES
    # ---------------------------------------
    async def _retrieve(self, ctx: Context, query: str, route: str, note: str) -> None:
        nodes = await retriever.aretrieve(query)
        existing = await sget(ctx, "retrieved_docs", [])
        await sset(ctx, "retrieved_docs", list(existing) + [n.get_content() for n in nodes])
        await sset(ctx, "route", route)
        await slog(ctx, note)

    @step
    async def retrieve_cooling_docs(
        self, ctx: Context, ev: RetrieveCoolingEvent
    ) -> DiagnoseEvent:
        await self._retrieve(
            ctx,
            "engine overheating coolant radiator fan thermostat diagnosis",
            "cooling",
            "Retrieved cooling-system documents.",
        )
        return DiagnoseEvent()

    @step
    async def retrieve_brake_docs(
        self, ctx: Context, ev: RetrieveBrakeEvent
    ) -> DiagnoseEvent:
        await self._retrieve(
            ctx,
            "low brake pressure hydraulic brake safety escalation immobilization",
            "brake",
            "Retrieved brake-safety documents.",
        )
        return DiagnoseEvent()

    @step
    async def retrieve_general_docs(
        self, ctx: Context, ev: RetrieveGeneralEvent
    ) -> DiagnoseEvent:
        await self._retrieve(
            ctx,
            "general automotive telemetry diagnosis",
            "general",
            "Retrieved general documents.",
        )
        return DiagnoseEvent()

    # ---------------------------------------
    # NODE + ROUTER: LLM DIAGNOSIS
    # ---------------------------------------
    @step
    async def llm_diagnosis(
        self, ctx: Context, ev: DiagnoseEvent
    ) -> DiagnoseEvent | RetrieveMoreEvent | HumanApprovalEvent | ReportEvent:
        telemetry = await sget(ctx, "telemetry", {})
        flags = await sget(ctx, "risk_flags", [])
        retrieved = await sget(ctx, "retrieved_docs", [])
        retry_count = await sget(ctx, "retry_count", 0)

        prompt = f"""
You are an automotive diagnostic assistant.

Telemetry:
{json.dumps(telemetry, indent=2)}

Risk flags:
{flags}

Retrieved maintenance context:
{chr(10).join(retrieved)}

Give output ONLY as JSON:
{{
  "diagnosis": "...",
  "confidence": 0.0,
  "recommended_action": "..."
}}

Rules:
- If brake pressure is low, mention safety escalation.
- If engine temperature is high, mention cooling system diagnosis.
- If vehicle is dangerous, recommend safe stop and immobilization approval.
"""

        try:
            response = await Settings.llm.acomplete(prompt)
            raw = str(response).strip()
            raw = raw.replace("```json", "").replace("```", "").strip()
            parsed = json.loads(raw)

            await sset(ctx, "diagnosis", parsed.get("diagnosis", "No diagnosis produced."))
            await sset(ctx, "confidence", float(parsed.get("confidence", 0.5)))
            await sset(ctx, "action", parsed.get("recommended_action", "No action."))
            await slog(ctx, "LLM diagnosis completed.")

        except Exception as exc:
            retry_count += 1
            await sset(ctx, "retry_count", retry_count)
            await sset(ctx, "diagnosis", f"LLM failed: {exc}")
            await sset(ctx, "confidence", 0.0)
            await slog(ctx, f"LLM failure. Retry count={retry_count}")

        # Router: retry / retrieve more / escalate / report
        confidence = await sget(ctx, "confidence", 0.0)
        retry_count = await sget(ctx, "retry_count", 0)

        if confidence == 0.0 and retry_count < 2:
            return DiagnoseEvent()

        if confidence < 0.70 and retry_count < 3:
            return RetrieveMoreEvent()

        if await sget(ctx, "safety_critical", False):
            return HumanApprovalEvent()

        return ReportEvent()

    # ---------------------------------------
    # NODE: RETRIEVE MORE DOCUMENTS IF CONFIDENCE LOW
    # ---------------------------------------
    @step
    async def retrieve_more_docs(
        self, ctx: Context, ev: RetrieveMoreEvent
    ) -> DiagnoseEvent:
        flags = await sget(ctx, "risk_flags", [])
        telemetry = await sget(ctx, "telemetry", {})
        await sset(ctx, "retry_count", await sget(ctx, "retry_count", 0) + 1)

        await self._retrieve(
            ctx,
            f"More details for diagnostic uncertainty. Flags: {flags} Telemetry: {telemetry}",
            await sget(ctx, "route", "general"),
            "Confidence low. Retrieved more documents and looping back to LLM.",
        )
        return DiagnoseEvent()

    # ---------------------------------------
    # NODE + ROUTER: HUMAN APPROVAL
    # ---------------------------------------
    @step
    async def human_approval(
        self, ctx: Context, ev: HumanApprovalEvent
    ) -> ImmobilizeEvent | ReportEvent:
        print("\nSAFETY CRITICAL CONDITION DETECTED")
        print("Vehicle:", await sget(ctx, "vehicle_id"))
        print("Telemetry:", await sget(ctx, "telemetry"))
        print("Diagnosis:", await sget(ctx, "diagnosis"))
        print("Recommended action:", await sget(ctx, "action"))

        if HUMAN_APPROVAL_MODE == "auto_yes":
            approved = True
        elif HUMAN_APPROVAL_MODE == "auto_no":
            approved = False
        else:
            try:
                answer = await asyncio.to_thread(
                    input, "\nApprove immobilization? Type YES or NO: "
                )
                approved = answer.strip().upper() == "YES"
            except Exception as exc:
                print(f"(interactive input unavailable: {exc}; defaulting to NO)")
                approved = False

        await sset(ctx, "human_approval_required", True)
        await sset(ctx, "human_approved", approved)
        await slog(ctx, f"Human approval captured: {approved}")

        return ImmobilizeEvent() if approved else ReportEvent()

    # ---------------------------------------
    # NODE: IMMOBILIZE VEHICLE
    # ---------------------------------------
    @step
    async def immobilize_vehicle(self, ctx: Context, ev: ImmobilizeEvent) -> ReportEvent:
        await sset(
            ctx,
            "action",
            "IMMOBILIZE VEHICLE AFTER SAFE STOP. Notify fleet control and service team.",
        )
        await slog(ctx, "Vehicle immobilization command issued.")
        return ReportEvent()

    # ---------------------------------------
    # NODE: FINAL REPORT
    # ---------------------------------------
    @step
    async def final_report(self, ctx: Context, ev: ReportEvent) -> StopEvent:
        await slog(ctx, "Final report generated.")
        state_log = await sget(ctx, "state_log", [])

        report = chr(10).join(
            [
                "AUTOMOTIVE TELEMETRY DIAGNOSTIC REPORT",
                "",
                f"Vehicle ID: {await sget(ctx, 'vehicle_id')}",
                "",
                "Telemetry:",
                json.dumps(await sget(ctx, "telemetry", {}), indent=2),
                "",
                f"Route: {await sget(ctx, 'route')}",
                f"Risk Flags: {await sget(ctx, 'risk_flags')}",
                "",
                "Diagnosis:",
                str(await sget(ctx, "diagnosis")),
                "",
                f"Confidence: {await sget(ctx, 'confidence')}",
                f"Retries: {await sget(ctx, 'retry_count')}",
                f"Safety Critical: {await sget(ctx, 'safety_critical')}",
                f"Human Approval Required: {await sget(ctx, 'human_approval_required')}",
                f"Human Approved: {await sget(ctx, 'human_approved')}",
                "",
                "Final Action:",
                str(await sget(ctx, "action")),
                "",
                "Execution Log:",
                chr(10).join(state_log),
            ]
        )
        return StopEvent(result=report)


print("workflow defined")

workflow defined


In [ ]:
workflow = TelemetryDiagnosticWorkflow(timeout=None, verbose=False)

report = await workflow.run(
    vehicle_id="KA01-FLEET-009",
    telemetry={
        "engine_temp_c": 118,
        "brake_pressure_bar": 38,
        "vehicle_speed_kmph": 62,
        "battery_voltage": 12.1,
        "coolant_level_percent": 24,
    },
)

print("\n" + "=" * 60)
print(report)

  [log] Stored initial vehicle state: {"vehicle_id": "KA01-FLEET-009", "telemetry": {"engine_temp_c": 118, "brake_pressure_bar": 38, "vehicle_speed_kmph": 62, "battery_voltage": 12.1, "coolant_level_percent": 24}}
  [log] Telemetry analyzed. Flags=['ENGINE_TEMP_HIGH', 'BRAKE_PRESSURE_LOW', 'DANGEROUS_BRAKE_WHILE_MOVING'], safety_critical=True
  [log] Retrieved brake-safety documents.
  [log] LLM diagnosis completed.

SAFETY CRITICAL CONDITION DETECTED
Vehicle: KA01-FLEET-009
Telemetry: {'engine_temp_c': 118, 'brake_pressure_bar': 38, 'vehicle_speed_kmph': 62, 'battery_voltage': 12.1, 'coolant_level_percent': 24}
Diagnosis: Multiple critical safety issues detected. The engine temperature is dangerously high (118°C), indicating a severe cooling system malfunction, potentially exacerbated by the low coolant level (24%). Concurrently, the brake hydraulic pressure is critically low (38 bar) while the vehicle is in motion (62 kmph), which severely compromises braking performance and presents

In [ ]:
workflow2 = TelemetryDiagnosticWorkflow(timeout=None, verbose=False)

report2 = await workflow2.run(
    vehicle_id="KA01-FLEET-014",
    telemetry={
        "engine_temp_c": 108,
        "brake_pressure_bar": 72,
        "vehicle_speed_kmph": 40,
        "battery_voltage": 12.6,
        "coolant_level_percent": 61,
    },
)

print("\n" + "=" * 60)
print(report2)